# 02. Field 制約と ValidationError の構造理解

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/ja/02_fields_and_errors.ipynb)

`int` や `str` といった型ヒントだけでも基本的な型安全性は確保できますが、実務のデータモデルには「数量は正の整数」「SKU コードの正規表現」「限定された配送区分」「インスタンス間で共有されない初期リスト」「camelCase の JSON キー変換」といったドメイン固有の制約が不可欠です。Pydantic では `Field(...)`・`typing.Annotated`・`typing.Literal` を用いてこれらのルールを宣言的に付与でき、Rust 製コア（`pydantic-core`）が**全フィールドを1パスで一括検証**するため、最初のエラーで途中終了せずにすべての違反箇所を一度に収集できます。

![図2-1: 宣言的な Field 制約カテゴリと一括エラー収集（ValidationError）の仕組み](https://examples.drawlib.com/pydantic/images/02_fields_and_errors/ch02_field_constraints_ja.png)

*図2-1: 宣言的な Field 制約カテゴリと一括エラー収集（ValidationError）の仕組み*

---

## 1. `Field` と `Literal` による宣言的なフィールド制約

数値の上下限（`gt`, `ge`, `lt`, `le`）、文字列長（`min_length`, `max_length`）、正規表現（`pattern`）は `Field(...)` でフィールドに直接宣言します。複数モデルで共通利用する制約付き型には `Annotated[T, Field(...)]`、取り得る選択肢が決まっている属性には `Literal[...]` を組み合わせるのが定石です。

また、以下の `tags: list[str] = Field(default_factory=list)` に注目してください。ミュータブルな空リストの共有バグを防ぐため、`default_factory=list` を指定することで `OrderItem` の生成ごとに独立した新しい `list` インスタンスが作られます。

In [ ]:
!pip install -q pydantic

In [ ]:
from typing import Annotated, Literal
from pydantic import BaseModel, ConfigDict, Field

SkuCode = Annotated[str, Field(pattern=r"^[A-Z]{3}-\d{4}$")]


class OrderItem(BaseModel):
    sku: SkuCode = "ORD-1024"
    name: str = Field(min_length=3, max_length=40)
    quantity: int = Field(gt=0, le=1000)
    unit_price: float = Field(ge=0.0)
    tier: Literal["standard", "express"] = "standard"
    tags: list[str] = Field(default_factory=list)


item = OrderItem(
    sku="PRO-4090",
    name="Mechanical Keyboard",
    quantity=2,
    unit_price=129.50,
    tier="express",
)
item.tags.append("fragile")

second_item = OrderItem(name="USB-C Cable", quantity=1, unit_price=12.00)
print("Validated item:", item.model_dump())
print("Second item tags (isolated list):", second_item.tags)

外部の REST / GraphQL API では `camelCase`（`itemQty`, `unitPrice`）の JSON キーが使われる一方、Python コード内では `snake_case`（`item_qty`, `unit_price`）で統一したい場面が頻繁にあります。`Field(alias=...)` と `ConfigDict(populate_by_name=True)` を併用すれば、入力時は**両方のキー名**を受け付けつつ、出力時は `model_dump(by_alias=True)` で外部向けの `camelCase` 辞書へ自在に変換できます。

In [ ]:
class ApiOrderItem(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    item_qty: int = Field(gt=0, le=1000, alias="itemQty")
    unit_price: float = Field(ge=0.0, alias="unitPrice")


# 1. 外部 API からの camelCase JSON ペイロードを検証
from_json = ApiOrderItem.model_validate({"itemQty": 4, "unitPrice": 19.95})

# 2. Python コード内では snake_case の属性名で直接初期化
from_python = ApiOrderItem(item_qty=2, unit_price=49.00)

print("Python snake_case dump:", from_json.model_dump())
print("External camelCase dump:", from_python.model_dump(by_alias=True))

---

## 2. `ValidationError` の内部構造（`err.errors()`）

手続き型の `if ... raise ValueError(...)` チェックでは最初の不正フィールドで処理が中断されますが、Pydantic は全フィールドを最後まで走査し、すべての違反を1つの `ValidationError` に集約します。`err.errors()` を呼び出すと、各違反の `loc`（発生箇所タプル）・`type`（機械可読エラーコード）・`msg`（人間向け説明文）・`input`（拒否された入力値）を格納した `list[dict]` が得られ、HTTP `422 Unprocessable Entity` のエラーレスポンスにそのまま活用できます。

![図2-2: API エラーレスポンスにそのまま使える ValidationError.errors() の構造](https://examples.drawlib.com/pydantic/images/02_fields_and_errors/ch02_validation_error_anatomy_ja.png)

*図2-2: API エラーレスポンスにそのまま使える ValidationError.errors() の構造*

実際に3つの制約違反（`name=""`, `quantity=0`, `tier="overnight"`）を同時に含む不正データを `OrderItem` に渡し、`ValidationError` を捕捉して `err.error_count()` と `err.errors()` の各フィールド（`loc`, `type`, `msg`）を整形出力してみましょう。

In [ ]:
from pydantic import ValidationError

try:
    OrderItem(
        name="",
        quantity=0,
        unit_price=29.99,
        tier="overnight",
    )
except ValidationError as err:
    print(f"Collected {err.error_count()} field errors in a single pass:")
    for e in err.errors():
        print((e["loc"], e["type"], e["msg"]))

    api_422_body = {
        "status": 422,
        "errors": [
            {
                "field": ".".join(str(part) for part in e["loc"]),
                "code": e["type"],
                "message": e["msg"],
                "input": e["input"],
            }
            for e in err.errors()
        ],
    }
    print("\nAPI 422 response payload:", api_422_body)